# Phase 6 - full grid (Colab GPU)

15 seeds x {linear, mlp} x 10 conditions x 3 datasets = **900 cells**.

**Set a GPU runtime** (Runtime -> Change runtime type -> T4 / L4 / A100). `u`/`v`
are held on the GPU and features are built per batch in torch, so the block
math is nearly free - ~1-2 h for the whole grid (vs hours-per-cell on CPU).

`--out` writes straight to a Drive parquet, flushed after every cell, so a
disconnect loses nothing - re-run the last cell to resume. All 900 cells run
on one device for consistency (the 90 CPU pilot cells are *not* reused).

In [ ]:
!git clone https://github.com/ryanteachman/sbert-head-ablation.git
%cd sbert-head-ablation
!pip install -q pyarrow pyyaml scikit-learn
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
assert torch.cuda.is_available(), 'set a GPU runtime: Runtime -> Change runtime type'
!nvidia-smi -L

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/sbert-head-ablation'
OUT = f'{DRIVE}/results/runs.parquet'
import os, json
os.makedirs(f'{DRIVE}/results', exist_ok=True)
!mkdir -p embeddings && rsync -a "{DRIVE}/embeddings/" embeddings/
assert len(json.load(open('embeddings/meta.json'))['splits']) == 11, 'run embed_colab.ipynb first'
import pandas as pd
print('starting from', len(pd.read_parquet(OUT)) if os.path.exists(OUT) else 0, '/ 900 cells')

## 1. Quick check (2 cells)
Confirms the pipeline runs on the real embeddings + GPU before the full run.

In [ ]:
!python src/run_grid.py --embed-dir embeddings --out /content/_check.parquet \
  --datasets nli --conditions C3 --heads linear,mlp --seeds 3 --limit 2
import pandas as pd; print(pd.read_parquet('/content/_check.parquet')[['dataset','condition','head','seed','test_acc','test_macro_f1','wall_s']].to_string())

## 2. Full grid
Resumable - re-run this cell after any disconnect. One line per cell.

In [ ]:
!python src/run_grid.py --embed-dir embeddings --out "{OUT}"

In [ ]:
import pandas as pd
df = pd.read_parquet(OUT)
print(f'{len(df)} / 900 cells\n')
for h in ['linear', 'mlp']:
    sub = df[df.head == h]
    if sub.empty: continue
    print(f'--- {h} : mean test_acc ---')
    print(sub.pivot_table(index='dataset', columns='condition', values='test_acc', aggfunc='mean').round(4).to_string())
    print()
ceiling = int(((df.epochs_trained == df.epochs_trained.max()) & (~df.early_stopped)).sum())
print('epoch-ceiling hits:', ceiling, '| seeds:', sorted(df.seed.unique()))
if len(df) == 900:
    from google.colab import files; files.download(OUT)

## Done
Download `runs.parquet` (also in Drive under `results/`), drop it in `results/`
locally, commit. Phase 7 analysis runs locally off that file.